In [15]:
import pandas as pd
import numpy as np
import pycountry_convert as pc

from bokeh.plotting import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Set2

output_notebook()

Loading BokehJS ...

In [16]:
def country_to_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except KeyError:
        pass


def merge_datasets(df: pd.DataFrame) -> pd.DataFrame:
    df["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)
    df["continent"] = df["country"].apply(country_to_continent)
    df = df[~df.continent.isna()]

    hdi_df = pd.read_csv("../../data/human_development_index.csv")[["iso3", "hdi_2020"]]
    df = pd.merge(left=df, right=hdi_df, how="left", left_on="gid", right_on="iso3")
    df = df[~df.hdi_2020.isna()].rename(columns={"hdi_2020": "hdi"}).drop(columns="iso3")

    df['size'] = np.log1p(df['pixel_count'])

    return df


def plot_scatter(df: pd.DataFrame):
    continents = df['continent'].unique().tolist()
    palette = Set2[max(3, len(continents))]

    source = ColumnDataSource.from_df(df)


    fig = figure(
        x_axis_label='Human Development Index', 
        y_axis_label='F1',
        width=980
    )
    fig.scatter(
        x="hdi",
        y="f1", 
        source=source,
        alpha=0.8,
        color=factor_cmap('continent', palette=palette, factors=continents),
        legend_field='continent',
        size="size"
    )

    hover = HoverTool(tooltips=[
        ("Country", "@country"),
        ("F1", "@f1"),
        ("Pixel Count", "@pixel_count"),
    ])

    fig.legend.location = "bottom_right"

    fig.add_tools(hover)
    show(fig)

## 1% Threshold

In [24]:
df = pd.read_parquet("../../results/global_1_percent_thresh")
df = merge_datasets(df)
plot_scatter(df)

*size of the circles is `log(total pixel count)`

## 20% Threshold

In [25]:
df = pd.read_parquet("../../results/global_20_percent_thresh")
df = merge_datasets(df)
plot_scatter(df)

In [5]:
import datetime
import pygadm
import geopandas as gpd
from conflict_monitoring_ntl.satellites import BlackMarblePy, GHSLSurface
from rasterio.enums import Resampling

from conflict_monitoring_ntl.transform import RasterPipeline
from conflict_monitoring_ntl.utils import get_combined_mask

rasters = [GHSLSurface(), BlackMarblePy(frequency="monthly")]
transformations = [{"reproject_match": {"resampling": Resampling.sum}}, {}]
date = datetime.date(2020, 1, 1)

gdf = pygadm.Items(name="Estonia", content_level=0)
gdf = gpd.GeoDataFrame(gdf.iloc[[0]].geometry).set_crs("EPSG:4326")

pipeline = RasterPipeline(gdf, date, rasters, transformations)
ds = pipeline.run()

mask = get_combined_mask(ds)
ds = ds.where(mask)

2025-11-12 09:41:22,792 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-12 09:41:22,808 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-12 09:41:22,854 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-12 09:41:22,871 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-12 09:41:23,898 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-12 09:41:25,968 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.goo

OBTAINING MANIFEST...:   0%|          | 0/1 [00:00<?, ?it/s]

2025-11-12 09:41:31,377 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files?product=VNP46A3&collection=5200&dateRanges=2020-01-01..2020-01-01&areaOfInterest=x21.76y57.51%2Cx28.21y59.82 "HTTP/1.1 200 OK"


QUEUEING TASKS | Downloading (83.4 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (83.4 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

2025-11-12 09:41:32,336 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A3/2020/001/VNP46A3.A2020001.h20v03.002.2025133144935.h5 "HTTP/1.1 200 OK"


  0%|          | 0.00/83.4M [00:00<?, ?B/s]

COLLECTING RESULTS | Downloading (83.4 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

COLLATING TILES | Processing...:   0%|          | 0/1 [00:00<?, ?date/s]

/Users/jan.kokla/Documents/EPFL/conflict-monitoring-ntl/src/conflict_monitoring_ntl/transform.py:97: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge(processed).rio.write_crs("EPSG:4326")


In [6]:
from conflict_monitoring_ntl.utils import binarize_xarray


ghsl_pop_binary_1pct = binarize_xarray(ds.ghsl_surface, 2500)
ghsl_pop_binary_20pct = binarize_xarray(ds.ghsl_surface, 50000)

bm_binary = binarize_xarray(ds.black_marble_radiance_monthly, 1.0)

In [ ]:
from conflict_monitoring_ntl.viz import plot_tile_comparison


plot_tile_comparison(
    arr_left=bm_binary, 
    arr_right=ghsl_pop_binary_1pct, 
    title_left="Black Marble Binary", 
    title_right="GHSL Surface 1% Threshold"
)

:Layout
   .Image.I  :Image   [lon,lat]   (black_marble_radiance_monthly)
   .Image.II :Image   [lon,lat]   (ghsl_surface)

In [ ]:
plot_tile_comparison(
    arr_left=bm_binary, 
    arr_right=ghsl_pop_binary_20pct, 
    title_left="Black Marble Binary", 
    title_right="GHSL Surface 20% Threshold"
)

:Layout
   .Image.I  :Image   [lon,lat]   (black_marble_radiance_monthly)
   .Image.II :Image   [lon,lat]   (ghsl_surface)

In [11]:
from sklearn.metrics import f1_score
from conflict_monitoring_ntl.utils import get_non_nan_flat_array


y_true = get_non_nan_flat_array(ghsl_pop_binary_1pct)
y_pred = get_non_nan_flat_array(bm_binary)

print(f1_score(y_true, y_pred))

0.4437048523126962


In [12]:
from sklearn.metrics import f1_score
from conflict_monitoring_ntl.utils import get_non_nan_flat_array


y_true = get_non_nan_flat_array(ghsl_pop_binary_20pct)
y_pred = get_non_nan_flat_array(bm_binary)

print(f1_score(y_true, y_pred))

0.013051257762340807


In [ ]:
from sklearn.metrics import confusion_matrix
from tqdm import tqdm


gdf = pygadm.Items(name="Estonia", content_level=1)
        
pixels = 0
conf_mat = np.zeros(4, dtype=np.int64)

province_ghsl_xarrs = []
province_bm_xarrs = []

for i in tqdm(range(len(gdf)), desc="Processing Provinces"):

    province_gdf = gpd.GeoDataFrame(gdf.iloc[[i]].geometry).set_crs("EPSG:4326")

    pipeline = RasterPipeline(province_gdf, date, rasters, transformations)
    ds = pipeline.run()

    # make sure we compare non-nan areas
    mask = get_combined_mask(ds)
    ds = ds.where(mask)

    ghsl_pop_binary = binarize_xarray(ds.ghsl_surface, 50000)
    y_true = get_non_nan_flat_array(ghsl_pop_binary)

    bm_binary = binarize_xarray(ds.black_marble_radiance_monthly, 1.0)
    y_pred = get_non_nan_flat_array(bm_binary)

    province_ghsl_xarrs.append(ghsl_pop_binary)
    province_bm_xarrs.append(bm_binary)

    assert y_pred.shape == y_true.shape

    conf_mat += confusion_matrix(y_true, y_pred).flatten()

In [42]:
TN, FP, FN, TP = conf_mat
(2 * TP) / (2 * TP + FP + FN)

np.float64(0.43618548164504556)

In [43]:
plot_tile_comparison(
    arr_left=province_bm_xarrs[0], 
    arr_right=province_ghsl_xarrs[0], 
    title_left="Black Marble Binary", 
    title_right="GHSL Surface 20% Threshold"
)

:Layout
   .Image.I  :Image   [lon,lat]   (black_marble_radiance_monthly)
   .Image.II :Image   [lon,lat]   (ghsl_surface)

In [62]:
conf_mat = np.zeros(4, dtype=np.int64)

for ghsl_pop, bm in zip(province_ghsl_xarrs, province_bm_xarrs):
    y_true = get_non_nan_flat_array(ghsl_pop)
    y_pred = get_non_nan_flat_array(bm)

    conf_mat += confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

TN, FP, FN, TP = conf_mat
(2 * TP) / (2 * TP + FP + FN)

np.float64(0.013263157894736841)